# ADOFAI Chart Generation Training (v1)

Train an AI model to generate ADOFAI (A Dance of Fire and Ice) charts from audio.

**Status**: Foundation training pipeline. Uses proof-of-concept LSTM model (Whisper integration TODO).

**Requirements**:
- ADOFAI charts in Google Drive (`level.adofai` + audio)
- GPU runtime (free tier: ~12 hours, Pro: longer)
- At least 100 charts recommended for initial quality

**Note**: Pretrained osu! weights are NOT compatible with ADOFAI. This trains from scratch.

## 1. Check GPU Runtime

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  No GPU detected. Training will be very slow on CPU.")
    print("   Go to Runtime → Change runtime type → GPU")

## 2. Configuration

In [ ]:
# === Configuration ===

# GitHub repo and branch
REPO_URL = "https://github.com/Tiller431/Mapperatorinator-ADOFAI.git"
BRANCH = "cursor/adofai-foundation-5317"  # Change to 'main' once PR is merged

# Google Drive dataset paths
ARCHIVE_PATH = "/content/drive/MyDrive/adofai-dataset/adofai-top100.tar.gz"  # Archive location
DRIVE_EXTRACT_DIR = "/content/drive/MyDrive/adofai-dataset/charts-top100"    # Extracted charts on Drive
LOCAL_EXTRACT_DIR = "/content/adofai-dataset/charts-top100"                  # Extracted charts on local disk

# Data directory (will be set after extraction)
DATA_DIR = None  # Auto-set in extraction cell

# Output directory (checkpoints saved to Drive)
OUTPUT_DIR = "/content/drive/MyDrive/adofai-checkpoints"

# Extraction strategy
# 'drive' = extract to Drive (slower but persistent)
# 'local' = extract to Colab disk (faster but lost on disconnect)
EXTRACT_TO = "local"  # Recommended: 'local' for speed

# Training settings
# NOTE: Use batch_size=1 or 2 for T4 GPU to avoid OOM (audio is now spectrogram-based)
BATCH_SIZE = 2        # Safe default for T4 GPU (14.56 GB memory)
LEARNING_RATE = 1e-4
EPOCHS = 50           # For full training
MAX_SAMPLES = None    # None = use all charts; set to number for limited run

# Smoke test settings (quick validation)
SMOKE_MODE = False    # Set to True for quick test

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"✓ Configuration set")
print(f"  Archive: {ARCHIVE_PATH}")
print(f"  Extract to: {EXTRACT_TO}")
print(f"  Output dir: {OUTPUT_DIR}")
print(f"  Device: {DEVICE}")
print(f"  Batch size: {BATCH_SIZE} (T4-safe)")
print(f"  Epochs: {EPOCHS}")
print(f"  Audio: 60s cap + log-mel spectrogram (80 mels)")

## 3. Mount Google Drive

This mounts your Google Drive to access:
- Dataset archive: `MyDrive/adofai-dataset/adofai-top100.tar.gz`
- Or extracted charts: `MyDrive/adofai-dataset/charts-top100/`
- Checkpoint storage: `MyDrive/adofai-checkpoints/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive mounted")

# Create output directory
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✓ Output directory ready: {OUTPUT_DIR}")

## 4. Extract Dataset

This cell handles dataset extraction:
1. If `charts-top100/` already exists, use it
2. Otherwise, extract from `adofai-top100.tar.gz`

**Extraction strategies**:
- `local` (recommended): Extract to Colab disk (`/content/`) — fast but lost on disconnect
- `drive`: Extract to Drive — persistent but slower I/O

Set `EXTRACT_TO` in config cell above.

In [ ]:
import os
import tarfile
from pathlib import Path

# Determine data directory based on extraction strategy
if EXTRACT_TO == "local":
    target_dir = LOCAL_EXTRACT_DIR
else:
    target_dir = DRIVE_EXTRACT_DIR

# Check if already extracted
if os.path.exists(target_dir):
    chart_dirs = [d for d in os.listdir(target_dir) if os.path.isdir(os.path.join(target_dir, d))]
    if len(chart_dirs) > 0:
        print(f"✓ Found existing dataset: {target_dir}")
        print(f"  {len(chart_dirs)} chart directories")
        print(f"  Examples: {chart_dirs[:3]}")
        DATA_DIR = target_dir
    else:
        print(f"⚠️  Directory exists but is empty: {target_dir}")
        print("   Will extract from archive...")
        DATA_DIR = None
else:
    DATA_DIR = None

# Extract if needed
if DATA_DIR is None:
    if not os.path.exists(ARCHIVE_PATH):
        print(f"❌ Archive not found: {ARCHIVE_PATH}")
        print(f"   Please upload adofai-top100.tar.gz to:")
        print(f"   MyDrive/adofai-dataset/")
        print(f"   Or use the shared folder: https://drive.google.com/drive/folders/1lATJxQI8P3uLsRtiC7ay5u3SrFhH1cfd")
        raise FileNotFoundError(ARCHIVE_PATH)
    
    print(f"📦 Extracting {ARCHIVE_PATH}...")
    print(f"   Target: {target_dir}")
    print(f"   This may take 2-5 minutes...\n")
    
    # Create parent directory
    os.makedirs(os.path.dirname(target_dir), exist_ok=True)
    
    # Extract
    with tarfile.open(ARCHIVE_PATH, 'r:gz') as tar:
        # Get total files for progress
        members = tar.getmembers()
        total = len(members)
        
        # Extract with progress
        for i, member in enumerate(members):
            tar.extract(member, path=os.path.dirname(target_dir))
            if (i + 1) % 100 == 0 or i == total - 1:
                print(f"  Extracted {i+1}/{total} files ({(i+1)/total*100:.1f}%)")
    
    print(f"\n✓ Extraction complete!")
    
    # Verify extraction
    if os.path.exists(target_dir):
        chart_dirs = [d for d in os.listdir(target_dir) if os.path.isdir(os.path.join(target_dir, d))]
        print(f"  Found {len(chart_dirs)} chart directories")
        if len(chart_dirs) > 0:
            print(f"  Examples: {chart_dirs[:3]}")
        DATA_DIR = target_dir
    else:
        print(f"❌ Extraction failed: {target_dir} not found")
        raise FileNotFoundError(target_dir)

print(f"\n✓ Dataset ready: {DATA_DIR}")

## 5. Clone Repository and Install Dependencies

In [ ]:
# Clone repo
!rm -rf Mapperatorinator-ADOFAI  # Remove if exists
!git clone -b {BRANCH} {REPO_URL}
%cd Mapperatorinator-ADOFAI

print(f"\n✓ Cloned {REPO_URL} (branch: {BRANCH})")

In [ ]:
# Install minimal dependencies for training
# Note: Skip heavy deps not needed for ADOFAI training

print("Installing dependencies...")
print("(This may take 2-3 minutes)\n")

# Core dependencies for ADOFAI training
!pip install -q pydub tqdm

# PyTorch should already be installed in Colab
# but ensure numpy is available
!pip install -q numpy

print("\n✓ Dependencies installed")

## 6. Verify Dataset Loading

In [ ]:
# Test dataset loading
from adofai.dataset import AdofaiDataset
from adofai.tokenizer import AdofaiTokenizer

print("Loading dataset (train split)...")
test_dataset = AdofaiDataset(
    data_dir=DATA_DIR,
    split='train',
    max_samples=5,  # Just load 5 to verify
)

samples = list(test_dataset)
print(f"\n✓ Successfully loaded {len(samples)} samples")

if len(samples) > 0:
    sample = samples[0]
    print(f"\nSample info:")
    print(f"  Chart: {sample['chart_name']}")
    print(f"  Audio shape: {sample['audio'].shape}")
    print(f"  BPM: {sample['bpm']}")
    print(f"  Events: {len(sample['events'])}")

# Initialize tokenizer
print("\nInitializing tokenizer...")
tokenizer = AdofaiTokenizer()
print(f"✓ Tokenizer ready (vocab_size={tokenizer.vocab_size})")

## 7. Smoke Test (Optional)

Run a quick training test with minimal data and a tiny model to verify everything works.

In [ ]:
# Smoke test: quick validation
# This trains on 5 samples for 2 epochs with a tiny model

if SMOKE_MODE or input("Run smoke test? (y/n): ").lower() == 'y':
    print("\n🔥 Running smoke test...\n")
    
    !python -m adofai.train \
        --data_dir {DATA_DIR} \
        --output_dir {OUTPUT_DIR}/smoke \
        --smoke \
        --device {DEVICE}
    
    print("\n✓ Smoke test complete!")
    print(f"  Checkpoints saved to {OUTPUT_DIR}/smoke")
else:
    print("Skipping smoke test")

## 8. Full Training

Train on your full dataset. This will take several hours depending on:
- Dataset size (100+ charts recommended)
- GPU type (T4/P100/V100)
- Number of epochs

**Colab Limits**:
- **Free tier**: ~12 hours per session (disconnect = lost progress if using local extraction)
- **Colab Pro**: Longer sessions, faster GPUs

**Tips**:
- Checkpoints save to Drive after each epoch
- If using `EXTRACT_TO="local"` and disconnected, dataset must be re-extracted (fast)
- If using `EXTRACT_TO="drive"`, dataset persists but training may be slower
- Monitor GPU usage: Runtime → Manage sessions

In [ ]:
# Full training
import time

print("Starting full training...")
print(f"  Dataset: {DATA_DIR}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Device: {DEVICE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Max samples: {MAX_SAMPLES or 'all'}")
print("\n" + "="*60 + "\n")

start_time = time.time()

# Build command
cmd = f"""python -m adofai.train \
    --data_dir {DATA_DIR} \
    --output_dir {OUTPUT_DIR} \
    --batch_size {BATCH_SIZE} \
    --lr {LEARNING_RATE} \
    --epochs {EPOCHS} \
    --device {DEVICE}"""

if MAX_SAMPLES is not None:
    cmd += f" --max_samples {MAX_SAMPLES}"

# Run training
!{cmd}

elapsed = time.time() - start_time
print("\n" + "="*60)
print(f"✓ Training complete!")
print(f"  Time: {elapsed/3600:.2f} hours")
print(f"  Checkpoints: {OUTPUT_DIR}")

## 9. Resume Training (If Interrupted)

If your session disconnects, re-run the setup cells above (mount Drive, extract dataset, clone repo), then run this cell to check checkpoint status.

**Note**: If you used `EXTRACT_TO="local"`, you'll need to re-extract the dataset (fast, ~2-5 min). Checkpoints are always on Drive.

In [ ]:
# Resume from checkpoint
import glob

checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint_epoch*.pt"))
if checkpoints:
    latest_checkpoint = checkpoints[-1]
    print(f"Found checkpoint: {latest_checkpoint}")
    
    # Extract epoch number
    import re
    match = re.search(r'epoch(\d+)', latest_checkpoint)
    if match:
        last_epoch = int(match.group(1))
        remaining_epochs = EPOCHS - last_epoch
        
        print(f"  Last completed epoch: {last_epoch}")
        print(f"  Remaining epochs: {remaining_epochs}")
        
        if remaining_epochs > 0:
            print("\nNote: Current training script doesn't support checkpoint resuming yet.")
            print("      You can restart training or adjust EPOCHS to continue.")
            print("      TODO: Add --checkpoint_path argument to train.py")
        else:
            print("\n✓ Training already complete!")
else:
    print("No checkpoints found. Run the full training cell above.")

## 10. Check Training Logs

View loss curves and checkpoint info.

In [ ]:
# List checkpoints
import glob
import torch

checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint_epoch*.pt"))
print(f"Found {len(checkpoints)} checkpoints:\n")

for ckpt in checkpoints:
    data = torch.load(ckpt, map_location='cpu')
    epoch = data.get('epoch', '?')
    loss = data.get('train_loss', '?')
    print(f"  Epoch {epoch+1}: loss={loss:.4f} - {ckpt}")

## 11. Next Steps

After training:

1. **Download checkpoints** from Google Drive to your local machine
2. **Test inference** with trained model (inference notebook TODO)
3. **Iterate**: Adjust hyperparameters, collect more data, train longer

**For production quality**:
- Integrate Whisper encoder from osuT5 (replace LSTM)
- Use spectrogram features instead of raw audio
- Train on 500+ charts
- Add evaluation metrics

See `docs/ADOFAI.md` in the repo for more details.

---

## Troubleshooting

**Archive not found**:
- Verify `adofai-top100.tar.gz` is in `MyDrive/adofai-dataset/`
- Or access shared folder: https://drive.google.com/drive/folders/1lATJxQI8P3uLsRtiC7ay5u3SrFhH1cfd
- Check `ARCHIVE_PATH` in config matches your Drive layout

**Extraction fails**:
- Drive may be full (check storage quota)
- Try `EXTRACT_TO="local"` instead (faster, uses Colab disk)
- Re-run extraction cell after fixing

**Out of Memory (OOM)**:
- Already optimized: audio capped at 60s, uses log-mel spectrogram (not raw waveforms)
- Default `BATCH_SIZE=2` is T4-safe; try 1 if still OOM
- Use CPU if GPU is too small (slow but works)

**No GPU**:
- Go to Runtime → Change runtime type → GPU (T4)
- Free tier: limited hours per day

**Training very slow**:
- If using `EXTRACT_TO="drive"`, try `"local"` (faster disk I/O)
- Verify GPU is enabled (check cell 1)
- Free tier GPUs are slower than Pro
- CPU training is ~10x slower

**Session disconnects**:
- Checkpoints save to Drive after each epoch
- Re-extract dataset if using `EXTRACT_TO="local"` (~2-5 min)
- Restart notebook from cell 1
- Colab Pro has longer sessions